# Linear Transformer for Blood Pressure Estimation

This notebook implements a Linear Transformer model for blood pressure estimation from pulse waveforms, with attention masks initialized from a pretrained LodeSTAR model.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
from tqdm import tqdm
import polars as pl
# import wandb  # Weights & Biases for experiment tracking
from sklearn.metrics import mean_squared_error
from LinearTransformerModel import LinearTransformer

%load_ext autoreload
%autoreload 2   


ModuleNotFoundError: No module named 'polars'

## Configuration

In [16]:
class Config:
    # Data parameters
    seq_len = 500
    batch_size = 32
    num_workers = 4
    
    # Model parameters
    input_dim = 2  # Assuming input has 2 channels (pulse and motion)
    embed_dim = 128
    num_heads = 8
    num_layers = 6
    hidden_dim = 512
    dropout = 0.1
    
    # Training parameters
    lr = 1e-4
    weight_decay = 1e-5
    epochs = 100
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Paths
    lodestar_weights_path = 'path_to_lodestar_weights.pth'
    checkpoint_dir = 'checkpoints'
    
    # Weights for loss components
    bp_loss_weight = 1.0
    pulse_loss_weight = 0.5
    
    # Initialize Weights & Biases
    use_wandb = True
    project_name = "bp_estimation_transformer"
    
config = Config()

# Create checkpoint directory
os.makedirs(config.checkpoint_dir, exist_ok=True)

'''Read controids trajectories from path to a numpy array

Input : str path to load

Output : np.array Dimension: [segmentLength, numCentroids, 2]  
'''
def read_from_csv(centroids_traj_fname):

    cents_pos = pd.read_csv(centroids_traj_fname)

    if 'Unnamed: 0' in cents_pos.columns:
        centroids = cents_pos.drop('Unnamed: 0', axis=1)
    else: 
        centroids  = cents_pos
    centroidSnaps     = centroids.values
    odd_columns       = centroidSnaps[:, 1::2]
    even_columns      = centroidSnaps[:, ::2]
    if odd_columns.shape != even_columns.shape:
        min_cols = min(odd_columns.shape[1], even_columns.shape[1])
        odd_columns  = odd_columns[:, :min_cols]
        even_columns = even_columns[:, :min_cols]
    centroid_trajectories = np.stack([odd_columns, even_columns], axis=2)   
    return centroid_trajectories

In [56]:
DataSegments = pd.read_csv(os.path.expanduser('~/DynoNAS/DatabaseTables/DataSegments.csv'))
display(DataSegments.head())

sid = '2024y_03m_14d_12h_14m_19s_787ms_358us_tracking_0'
            
cond1 = DataSegments['unique_file_id'] == sid.partition('us')[0] + sid.partition('us')[1]
cond2  = DataSegments['relative_idx'] == int(sid[-1])
cond3 = DataSegments['type'] == 'tracking'
subject_id = DataSegments[cond1 & cond2 & cond3]
display(subject_id)

,id,subject_id,record_id,nas_path,unique_file_id,visit_id,visit_type,hospital_name,study_name,ward_name,ground_truth_type,type,start_idx,end_idx,relative_idx,schema_version
0,1,1603,6,DynoNAS/Clinical_Studies/LSU/PID1603,2023y_09m_12d_14h_37m_38s_735ms_830us,2,in_patient,LSU,NaN,NaN,NaN,tracking,15051,20863,0,1
1,2,1603,6,DynoNAS/Clinical_Studies/LSU/PID1603,2023y_09m_12d_14h_37m_38s_735ms_830us,2,in_patient,LSU,NaN,NaN,NaN,upsweep,7238,8972,0,1
2,3,1603,6,DynoNAS/Clinical_Studies/LSU/PID1603,2023y_09m_12d_14h_37m_38s_735ms_830us,2,in_patient,LSU,NaN,NaN,NaN,downsweep,8975,10710,0,1
3,4,1602,7,DynoNAS/Clinical_Studies/LSU/PID1602,2023y_09m_12d_21h_25m_30s_138ms_525us,3,in_patient,LSU,NaN,NaN,NaN,tracking,52929,70437,0,1
4,5,1602,7,DynoNAS/Clinical_Studies/LSU/PID1602,2023y_09m_12d_21h_25m_30s_138ms_525us,3,in_patient,LSU,NaN,NaN,NaN,tracking,76698,88094,1,1


,id,subject_id,record_id,nas_path,unique_file_id,visit_id,visit_type,hospital_name,study_name,ward_name,ground_truth_type,type,start_idx,end_idx,relative_idx,schema_version
626,627,6,146,DynoNAS/Clinical_Studies/ClevelandClinic/CCF_D...,2024y_03m_14d_12h_14m_19s_787ms_358us,60,in_patient,ClevelandClinic,NaN,NaN,IAP,tracking,23555,36817,0,3


## Dataset and DataLoader

In [61]:
class BPDataset(Dataset):
    def __init__(self, points_data_path, BP_data_path, window_size =120, overlap =20, use_overlap=True, transform= None):
        """
        Args:
            data_path: Path to the dataset
            split: 'train', 'val', or 'test'
        """
        # Load your dataset here
        import glob
        self.centroids_dir = points_data_path
        self.bp_dir = BP_data_path
        self.window_size = window_size
        self.overlap = overlap
        self.use_overlap = use_overlap
        self.transform = transform
        self.stride = (int(window_size) - int(overlap)) if use_overlap else window_size

        # Identify unique subject IDs by checking files with a specific suffix
        files = os.listdir(self.centroids_dir)
        self.subject_ids = sorted([
            f.split("_r0.05_CentroidPositionsLowPass.csv")[0] 
            for f in files if f.endswith("_r0.05_CentroidPositionsLowPass.csv")
        ])

        self.samples = []

        for sid in self.subject_ids:
            print(sid)
            try: 
                clean_path = os.path.join(self.centroids_dir, f'{sid}_r0.05_CentroidPositionsLowPass.csv')
                easy_mixed_path = os.path.join(self.centroids_dir, f'{sid}_r0.35_CentroidPositionsLowPass.csv')
                hard_mixed_path = os.path.join(self.centroids_dir, f'{sid}_r0.65_CentroidPositionsLowPass.csv')
                noise_path = os.path.join(self.centroids_dir, f'{sid}_r0.95_CentroidPositionsLowPass.csv')
                
                cond1 = DataSegments['unique_file_id'] == sid.partition('us')[0] + sid.partition('us')[1]
                cond2  = DataSegments['relative_idx'] == int(sid[-1])
                cond3 = DataSegments['type'] == 'tracking'
                subject_id = DataSegments[cond1 & cond2 & cond3]['subject_id'].values[0]
                bp_path = os.path.join(self.bp_dir, f'{subject_id}_{sid}', f'{sid}_Time.csv')
                clean_data = read_from_csv(clean_path)
                easy_mixed_data = read_from_csv(easy_mixed_path)
                hard_mixed_data = read_from_csv(hard_mixed_path)
                noise_data = read_from_csv(noise_path)
                bp_data = pd.read_csv(bp_path)['PULSE_Y'].to_numpy()


                # Find the minimum length among the triad to ensure synchronized windowing
                min_len = min(len(clean_data), len(noise_data), len(easy_mixed_data), len(hard_mixed_data))

                if self.transform:
                    clean_data = self.transform(clean_data)
                    easy_mixed_data = self.transform(easy_mixed_data)
                    hard_mixed_data = self.transform(hard_mixed_data)
                    noise_data = self.transform(noise_data)
                # Create segments using the calculated stride
                for start in range(0, min_len - window_size + 1, self.stride):
                    end = start + window_size
                    self.samples.append({
                        'clean': clean_data[start:end],
                        'noise': noise_data[start:end],
                        'mixed': easy_mixed_data[start:end],
                        'bp' : bp_data[start:end]
                    })
            except Exception as e:
                print(f"Error processing subject {sid}: {e}")

    def __len__(self):
        return len(self.subject_ids)
    
    def __getitem__(self, idx):
        smaple = self.samples[idx]

        clean_data = torch.tensor(smaple['clean'], dtype=torch.float32)
        easy_mixed_data = torch.tensor(smaple['easy_mixed'], dtype=torch.float32)
        hard_mixed_data = torch.tensor(smaple['hard_mixed'], dtype=torch.float32)
        noise_data = torch.tensor(smaple['noise'], dtype=torch.float32)
        bp = torch.tensor(smaple['bp'], dtype=torch.float32)

        return clean_data, easy_mixed_data, hard_mixed_data, noise_data, bp
        
# Create datasets and dataloaders
full_dataset = BPDataset(points_data_path= os.path.expanduser('~/DynoNAS/Peter/Research_data/Synthetic_Videos_Noise_Mixture_Protocol'),
                        BP_data_path= os.path.expanduser('~/DynoNAS/ProcessedData/BeatDetector_v3'))  
dataloader = DataLoader(full_dataset, batch_size=8, shuffle=True)

num_cpus = os.cpu_count() 

# 2. Split the dataset (0.7 Train, 0.3 Test)
train_size = int(0.7 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset,
                          batch_size=8,
                          shuffle=True,
                          num_workers=min(4, num_cpus), # Conservative start
                          pin_memory=True,              # Faster data transfer to GPU
                          persistent_workers=True       # Keeps workers alive between epochs for speed)
                        )

test_loader = DataLoader(test_dataset,
                         batch_size=8,
                         shuffle=False,
                         num_workers=min(4, num_cpus), # Conservative start
                         pin_memory=True,              # Faster data transfer to GPU
                         persistent_workers=True       # Keeps workers alive between epochs for speed)
                        )

2024y_03m_14d_12h_14m_19s_787ms_358us_tracking_0
2024y_03m_14d_12h_37m_39s_532ms_546us_tracking_0
2024y_03m_14d_12h_37m_39s_532ms_546us_tracking_1
2024y_03m_14d_13h_51m_42s_069ms_802us_tracking_0
2024y_03m_14d_13h_51m_42s_069ms_802us_tracking_1
2024y_03m_19d_07h_51m_21s_131ms_313us_tracking_0
2024y_03m_19d_08h_10m_35s_594ms_614us_tracking_0
2024y_03m_19d_08h_25m_28s_119ms_025us_tracking_0
Error processing subject 2024y_03m_19d_08h_25m_28s_119ms_025us_tracking_0: [Errno 2] No such file or directory: '/home/peter/DynoNAS/ProcessedData/BeatDetector_v3/7_2024y_03m_19d_08h_25m_28s_119ms_025us_tracking_0/2024y_03m_19d_08h_25m_28s_119ms_025us_tracking_0_Time.csv'
2024y_03m_19d_08h_51m_33s_507ms_570us_tracking_0
2024y_03m_19d_09h_38m_16s_473ms_453us_tracking_0
2024y_03m_19d_10h_47m_56s_562ms_123us_tracking_0
2024y_03m_19d_11h_38m_57s_737ms_755us_tracking_0
2024y_03m_19d_13h_02m_45s_911ms_259us_tracking_0
2024y_04m_04d_10h_59m_37s_291ms_241us_tracking_0
2024y_04m_10d_09h_46m_14s_945ms_318us_tra

## Model Initialization

In [24]:
def initialize_model():
    """Initialize the model and load pretrained attention masks if available"""
    model = LinearTransformer(
        input_dim=config.input_dim,
        embed_dim=config.embed_dim,
        num_heads=config.num_heads,
        num_layers=config.num_layers,
        hidden_dim=config.hidden_dim,
        dropout=config.dropout,
        max_seq_len=config.seq_len
    )
    
    # Load pretrained attention masks from LodeSTAR if available
    if os.path.exists(config.lodestar_weights_path):
        try:
            model.load_pretrained_attention_masks(config.lodestar_weights_path)
            print("Successfully loaded attention masks from LodeSTAR model")
        except Exception as e:
            print(f"Error loading attention masks: {e}")
    
    return model.to(config.device)

model = initialize_model()

## Training Setup

In [26]:
def setup_training(model):
    """Set up optimizer, scheduler, and loss functions"""
    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay
    )
    
    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
    )
    
    # Loss functions
    bp_criterion = nn.MSELoss()  # For BP estimation
    pulse_criterion = nn.MSELoss()  # For pulse waveform prediction
    
    return optimizer, scheduler, bp_criterion, pulse_criterion

optimizer, scheduler, bp_criterion, pulse_criterion = setup_training(model)

## Training Loop

In [27]:
def train_epoch(model, dataloader, optimizer, bp_criterion, pulse_criterion, epoch):
    model.train()
    total_loss = 0.0
    bp_loss_total = 0.0
    pulse_loss_total = 0.0
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Train]")
    
    for batch in progress_bar:
        # Move data to device
        inputs = batch['input'].to(config.device)
        bp_targets = batch['bp'].to(config.device)
        pulse_targets = batch['pulse'].to(config.device)
        
        # Forward pass
        optimizer.zero_grad()
        bp_pred, pulse_pred = model(inputs)
        
        # Calculate losses
        bp_loss = bp_criterion(bp_pred, bp_targets)
        pulse_loss = pulse_criterion(pulse_pred, pulse_targets)
        
        # Combine losses
        loss = config.bp_loss_weight * bp_loss + config.pulse_loss_weight * pulse_loss
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Update metrics
        total_loss += loss.item()
        bp_loss_total += bp_loss.item()
        pulse_loss_total += pulse_loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': total_loss / (progress_bar.n + 1),
            'bp_loss': bp_loss_total / (progress_bar.n + 1),
            'pulse_loss': pulse_loss_total / (progress_bar.n + 1)
        })
    
    # Calculate average losses
    avg_loss = total_loss / len(dataloader)
    avg_bp_loss = bp_loss_total / len(dataloader)
    avg_pulse_loss = pulse_loss_total / len(dataloader)
    
    # Log average losses
    print(f"Epoch {epoch+1} [Train] - Avg Loss: {avg_loss:.4f}, Avg BP Loss: {avg_bp_loss:.4f}, Avg Pulse Loss: {avg_pulse_loss:.4f}")
    
    return avg_loss, avg_bp_loss, avg_pulse_loss
